### Online Retail II — Standardize Categorical Variables, Normalize Numerical Features

Takes the cleaned dataset (`data/online_retail_ii_cleaned.parquet`) and prepares it for feature engineering (Task #4):

1. Load + inspect cleaned data
2. Standardize categorical variables (Country) via one-hot encoding
3. Check numerical feature distributions (skew)
4. Normalize numerical features (Quantity, Price, Total_Price) with log transform + z-score
5. Confirm InvoiceDate is proper datetime
6. Save the standardized/normalized dataset

## 1. Load + inspect cleaned data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

df = pd.read_parquet('../data/outliers_handled.parquet')
print(df.shape)
df.head()

(608568, 10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_Price,CustomerMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009-12


In [2]:
df.dtypes

Invoice                  string
StockCode                   str
Description              string
Quantity                  int64
InvoiceDate      datetime64[ns]
Price                   float64
Customer ID               int64
Country                  string
Total_Price             float64
CustomerMonth         period[M]
dtype: object

## 2. Standardize categorical variables

`Country` is the main categorical feature. It's high-cardinality (41 unique values) and heavily
skewed toward United Kingdom (~90% of rows), so rare countries are grouped into `"Other"` before
one-hot encoding to avoid creating dozens of near-empty columns.

In [3]:
# check current distribution
country_counts = df['Country'].value_counts()
print(f"Unique countries: {df['Country'].nunique()}")
country_counts.head(15)

Unique countries: 41


Country
United Kingdom     548514
Germany             12485
EIRE                12120
France               9422
Netherlands          4165
Spain                2668
Switzerland          2238
Belgium              2159
Portugal             1597
Australia            1526
Channel Islands      1300
Sweden               1113
Italy                1059
Cyprus                878
Finland               771
Name: count, dtype: int64[pyarrow]

In [4]:
# group rare countries (fewer than 0.5% of rows) into "Other"
threshold = len(df) * 0.005
top_countries = country_counts[country_counts >= threshold].index

df['Country_grouped'] = df['Country'].where(df['Country'].isin(top_countries), 'Other')
print(f"Countries kept individually: {df['Country_grouped'].nunique() - 1}")
df['Country_grouped'].value_counts()

Countries kept individually: 5


Country_grouped
United Kingdom    548514
Other              21862
Germany            12485
EIRE               12120
France              9422
Netherlands         4165
Name: count, dtype: int64[pyarrow]

In [5]:
# one-hot encode the grouped country column
country_dummies = pd.get_dummies(df['Country_grouped'], prefix='Country')
df = pd.concat([df, country_dummies], axis=1)

print(f"Added {country_dummies.shape[1]} one-hot columns")
df.filter(like='Country_').head()

Added 6 one-hot columns


,Country_grouped,Country_EIRE,Country_France,Country_Germany,Country_Netherlands,Country_Other,Country_United Kingdom
0,United Kingdom,False,False,False,False,False,True
1,United Kingdom,False,False,False,False,False,True
2,United Kingdom,False,False,False,False,False,True
3,United Kingdom,False,False,False,False,False,True
4,United Kingdom,False,False,False,False,False,True


## 3. Check numerical feature distributions

Retail transaction amounts are typically right-skewed (a few large orders pull the distribution),
so check skew before picking a normalization method.

In [6]:
num_cols = ['Quantity', 'Price', 'Total_Price']
print(df[num_cols].skew())

Quantity        4.465971
Price           1.835682
Total_Price    11.108316
dtype: float64


## 4. Normalize numerical features

All three numeric columns are heavily right-skewed (skew >> 1), so apply a log transform first
(`log1p` to safely handle values close to zero) and then z-score standardize, rather than
z-scoring the raw skewed values directly.

In [7]:
# log transform (log1p handles values near zero safely)
for col in num_cols:
    df[f'{col}_log'] = np.log1p(df[col])

print(df[[f'{c}_log' for c in num_cols]].skew())

Quantity_log       0.698754
Price_log          0.602026
Total_Price_log    0.319869
dtype: float64


In [8]:
# z-score standardize the log-transformed columns
scaler = StandardScaler()
log_cols = [f'{c}_log' for c in num_cols]
scaled_cols = [f'{c}_scaled' for c in num_cols]

df[scaled_cols] = scaler.fit_transform(df[log_cols])
df[scaled_cols].describe()

,Quantity_scaled,Price_scaled,Total_Price_scaled
count,6.085680e+05,6.085680e+05,6.085680e+05
mean,1.457121e-16,3.003911e-16,-3.766098e-16
std,1.000001e+00,1.000001e+00,1.000001e+00
min,-1.247193e+00,-2.034664e+00,-2.546700e+00
25%,-8.358706e-01,-6.278463e-01,-7.054472e-01
50%,2.366608e-02,-1.573473e-01,1.260196e-01
75%,6.516471e-01,6.700355e-01,5.612680e-01
max,3.098269e+00,2.516241e+00,5.151519e+00


## 5. Confirm InvoiceDate is proper datetime

In [9]:
# should already be datetime64[ns] from the cleaning notebook, confirm here
print(df['InvoiceDate'].dtype)
assert pd.api.types.is_datetime64_any_dtype(df['InvoiceDate']), "InvoiceDate is not datetime"
print("InvoiceDate confirmed as datetime")

datetime64[ns]
InvoiceDate confirmed as datetime


## 6. Save standardized/normalized dataset

In [ ]:
print("Final shape:", df.shape)
print("\nNew columns added:")
print([c for c in df.columns if c.endswith(('_log', '_scaled')) or c.startswith('Country_')])

df.to_parquet('../data/standardized.parquet', index=False)
print("\nSaved to data/standardized.parquet")

Final shape: (608568, 23)

New columns added:
['Country_grouped', 'Country_EIRE', 'Country_France', 'Country_Germany', 'Country_Netherlands', 'Country_Other', 'Country_United Kingdom', 'Quantity_log', 'Price_log', 'Total_Price_log', 'Quantity_scaled', 'Price_scaled', 'Total_Price_scaled']



Saved to data/online_retail_ii_standardized.parquet
